<a href="https://colab.research.google.com/github/Vihsacc/CP1_eng_software/blob/main/Eng_Softw.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
import json
import os
import hashlib

ARQ_USUARIOS = "usuarios.json"
ARQ_RESERVAS = "reservas.json"

usuario_logado = None

# ------------------ ARQUIVOS ------------------
def carregar_usuarios():
    if os.path.exists(ARQ_USUARIOS):
        with open(ARQ_USUARIOS, "r") as f:
            return json.load(f)
    return {}

def salvar_usuarios(usuarios):
    with open(ARQ_USUARIOS, "w") as f:
        json.dump(usuarios, f, indent=4)

def carregar_reservas():
    if os.path.exists(ARQ_RESERVAS):
        with open(ARQ_RESERVAS, "r") as f:
            return json.load(f)
    return {
        "Sala 1": [],
        "Sala 2": [],
        "Sala 3": []
    }

def salvar_reservas(salas):
    with open(ARQ_RESERVAS, "w") as f:
        json.dump(salas, f, indent=4)

usuarios = carregar_usuarios()
salas = carregar_reservas()

# ------------------ HASH SENHA ------------------
def criptografar_senha(senha):
    return hashlib.sha256(senha.encode()).hexdigest()

# ------------------ CADASTRO ------------------
def cadastrar():
    print("\n--- Cadastro ---")
    user = input("Novo usuário: ")

    if user in usuarios:
        print("❌ Usuário já existe!")
        return

    senha = input("Nova senha: ")
    usuarios[user] = criptografar_senha(senha)
    salvar_usuarios(usuarios)

    print("✅ Usuário cadastrado com sucesso!")

# ------------------ LOGIN ------------------
def login():
    global usuario_logado
    print("\n--- Login ---")

    user = input("Usuário: ")
    senha = input("Senha: ")

    senha_hash = criptografar_senha(senha)

    if user in usuarios and usuarios[user] == senha_hash:
        usuario_logado = user
        print(f"✅ Bem-vindo, {user}!")
        return True
    else:
        print("❌ Usuário ou senha inválidos!")
        return False

# ------------------ VALIDAÇÃO HORÁRIO ------------------
def horario_valido(horario):
    try:
        hora = int(horario.split(":")[0])
        return 6 <= hora <= 23
    except:
        return False

# ------------------ DISPONIBILIDADE ------------------
def ver_disponibilidade():
    print("\n--- Disponibilidade das Salas ---")

    for sala, reservas in salas.items():
        print(f"\n{sala}:")

        if reservas:
            reservas_ordenadas = sorted(reservas, key=lambda x: x["horario"])

            for r in reservas_ordenadas:
                print(f"  - {r['horario']} (Reservado por {r['nome']})")
        else:
            print("  Livre")

# ------------------ RESERVAR ------------------
def reservar():
    sala = input("Escolha a sala (Sala 1, Sala 2, Sala 3): ")
    horario = input("Digite o horário (HH:MM): ")

    if not horario_valido(horario):
        print("❌ Horário inválido! Use entre 06:00 e 23:00")
        return

    if sala in salas:
        for reserva in salas[sala]:
            if reserva["horario"] == horario:
                print("❌ Horário já reservado!")
                return

        salas[sala].append({
            "nome": usuario_logado,
            "horario": horario
        })

        salvar_reservas(salas)
        print("✅ Reserva feita com sucesso!")
    else:
        print("❌ Sala inválida!")

# ------------------ CANCELAR ------------------
def cancelar():
    sala = input("Digite a sala: ")
    horario = input("Digite o horário (HH:MM): ")

    if sala in salas:
        for reserva in salas[sala]:
            if reserva["nome"] == usuario_logado and reserva["horario"] == horario:
                salas[sala].remove(reserva)
                salvar_reservas(salas)
                print("✅ Reserva cancelada!")
                return

        print("❌ Reserva não encontrada ou não pertence a você!")
    else:
        print("❌ Sala inválida!")

# ------------------ HISTÓRICO ------------------
def minhas_reservas():
    print("\n--- Minhas Reservas ---")
    encontrou = False

    for sala, reservas in salas.items():
        for reserva in reservas:
            if reserva["nome"] == usuario_logado:
                print(f"{sala} - {reserva['horario']}")
                encontrou = True

    if not encontrou:
        print("Nenhuma reserva encontrada.")

# ------------------ MENU PRINCIPAL ------------------
def menu():
    while True:
        print("\n--- Sistema de Reserva ---")
        print("1. Ver disponibilidade")
        print("2. Reservar sala")
        print("3. Cancelar reserva")
        print("4. Minhas reservas")
        print("5. Sair")

        opcao = input("Escolha uma opção: ")

        if opcao == "1":
            ver_disponibilidade()
        elif opcao == "2":
            reservar()
        elif opcao == "3":
            cancelar()
        elif opcao == "4":
            minhas_reservas()
        elif opcao == "5":
            print("Saindo...")
            break
        else:
            print("❌ Opção inválida!")

# ------------------ MENU INICIAL ------------------
def iniciar():
    while True:
        print("\n--- Bem-vindo ---")
        print("1. Login")
        print("2. Cadastro")
        print("3. Sair")

        opcao = input("Escolha uma opção: ")

        if opcao == "1":
            if login():
                menu()
        elif opcao == "2":
            cadastrar()
        elif opcao == "3":
            print("Encerrando...")
            break
        else:
            print("❌ Opção inválida!")

# ------------------ EXECUÇÃO ------------------
iniciar()